In [231]:
import re
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, f1_score, confusion_matrix

np.random.seed(42)
random.seed(42)

In [232]:
# 1. Функция генерации датасета 
# !!!prefixes, suffixes, common_phrases, category_pools, ambiguous_phrases - сгенерировала нейросеть
def generate_bank_dataset(n_samples=1000):
    prefixes = [
        "Здравствуйте, ", "Подскажите, ", "Проблема: ", "Срочный вопрос, ",
        "Добрый день, ", "", "Не могу разобраться: "
    ]

    suffixes = [
        "!", " помогите решить.", " жду ответа.", " что делать?",
        " заранее спасибо.", "", " подскажите пожалуйста.", " нужна помощь."
    ]

    # Фразы, которые могут встречаться в любом классе
    common_phrases = [
        "не работает", "не получается", "возникла ошибка", "операция не проходит",
        "не отображается", "долго обрабатывается", "не могу выполнить операцию",
        "нужна помощь", "что делать", "проблема с операцией", "не могу разобраться",
        "списали деньги", "возникла проблема", "всё было нормально раньше",
        "не получается воспользоваться"
    ]

    # Cлова, частые для одной категории
    category_pools = {
        "Карты": [
            "карта", "банковская карта", "дебетовая карта", "банкомат",
            "пин код", "бесконтактная оплата", "лимит",
            "кэшбэк", "реквизиты", "виртуальная карта",
            "оплата", "снятие наличных", "пластик"
        ],

        "Кредиты": [
            "кредит", "кредитный счет", "платеж", "ежемесячный платеж",
            "ставка", "задолженность", "договор", "ипотека",
            "страховка", "просрочка", "остаток долга", "комиссия",
            "платеж по счету", "деньги"
        ],

        "Переводы": [
            "перевод", "деньги", "средства", "транзакция",
            "СБП", "платеж", "получатель", "реквизиты",
            "зачисление", "перевод на карту", "перевод в другой банк",
            "комиссия", "операция"
        ],

        "Мошенничество": [
            "неизвестная операция", "списание", "подозрительная операция",
            "мошенники", "взлом", "код из смс", "чужой платеж",
            "подозрительная активность", "злоумышленник",
            "доступ к аккаунту", "кража денег", "операция",
            "перевод денег", "списали средства"
        ],

        "Сервис": [
            "приложение", "поддержка", "оператор", "чат",
            "интернет банк", "личный кабинет", "уведомления",
            "вход", "авторизация", "офис", "сотрудник",
            "обновление", "ошибка", "работа приложения"
        ]
    }

    # Ситуации, в которых один и тот же объект встречается
    # в разных категориях
    ambiguous_phrases = {
        "Карты": [
            "не проходит оплата", "заблокировали", "списали деньги",
            "не могу воспользоваться картой", "операция отклонена",
            "не отображается операция"
        ],

        "Кредиты": [
            "не прошел платеж", "списали деньги", "не отображается платеж",
            "возникла ошибка", "не могу оплатить", "нужна информация по счету"
        ],

        "Переводы": [
            "операция не прошла", "деньги не пришли", "списали деньги",
            "возникла ошибка", "неизвестный статус", "не отображается операция"
        ],

        "Мошенничество": [
            "списали деньги", "неизвестная операция", "перевод денег",
            "операция не моя", "не могу понять списание",
            "деньги ушли со счета"
        ],

        "Сервис": [
            "возникла ошибка", "не работает", "не отображается",
            "операция не проходит", "не могу войти",
            "проблема в приложении"
        ]
    }

    data = []
    labels = list(category_pools.keys())

    for _ in range(n_samples):
        cat = random.choice(labels)

        # Количество категориальных фраз
        n_category = random.choices(
            [1, 2],
            weights=[0.7, 0.3]
        )[0]

        category_phrases = random.sample(
            category_pools[cat],
            k=n_category
        )

        # Общие фразы
        n_common = random.choices(
            [1, 2],
            weights=[0.7, 0.3]
        )[0]

        common = random.sample(
            common_phrases,
            k=n_common
        )
        ambiguous = []

        # С шансом 0.65 добавляем фразу из других блоков
        if random.random() < 0.65:
            ambiguous.append(random.choice(ambiguous_phrases[cat]))

        phrases = category_phrases + common + ambiguous
        random.shuffle(phrases)

        core_text = " ".join(phrases)

        # Вариативность структуры предложения
        templates = [
            "{}",
            "у меня {}",
            "такая ситуация: {}",
            "столкнулся с проблемой: {}",
            "хочу уточнить: {}",
            "не знаю что делать, {}",
            "пожалуйста помогите, {}"
        ]

        core_text = random.choice(templates).format(core_text)

        full_text = (
            f"{random.choice(prefixes)}"
            f"{core_text}"
            f"{random.choice(suffixes)}"
        )

        data.append({
            "text": full_text.strip(),
            "category": cat
        })

    return pd.DataFrame(data)

# Генерируем датасет
df = generate_bank_dataset(n_samples=2000)
print(df['category'].value_counts())
print("\n", df.head(5))

category
Кредиты          420
Карты            401
Мошенничество    400
Переводы         398
Сервис           381
Name: count, dtype: int64

                                                 text category
0  Срочный вопрос, хочу уточнить: возникла ошибка...    Карты
1  Срочный вопрос, у меня банкомат не работает не...    Карты
2  Проблема: не отображается снятие наличных спис...    Карты
3  Проблема: хочу уточнить: не получается приложе...   Сервис
4  Подскажите, пожалуйста помогите, операция не п...   Сервис


In [233]:
# 2. Предобработка текста и Split
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^а-яа-еa-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(preprocess_text)

label_mapping = {cat: idx for idx, cat in enumerate(df['category'].unique())}
inv_label_mapping = {v: k for k, v in label_mapping.items()}
df['label'] = df['category'].map(label_mapping)

# Split на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], 
    df['label'], 
    test_size=0.25, 
    random_state=42, 
    stratify=df['label']
)

print(f"Train length: {len(X_train)}")
print(f"Test length: {len(X_test)}")

Train length: 1500
Test length: 500


In [234]:
# 3. TF-IDF и Baseline
vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 1), min_df=2, max_df=0.95)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Модель 1
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)
nb_f1 = f1_score(y_test, nb_preds, average='macro')

# Модель 2
logreg_model = LogisticRegression(C=1, max_iter=1000)
logreg_model.fit(X_train_tfidf, y_train)
logreg_preds = logreg_model.predict(X_test_tfidf)
logreg_f1 = f1_score(y_test, logreg_preds, average='macro')

print(f"MultinomialNB F1-macro: {nb_f1:.4f}")
print(f"LogisticRegression F1-macro: {logreg_f1:.4f}")

MultinomialNB F1-macro: 0.9397
LogisticRegression F1-macro: 0.9434


In [235]:
import collections
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report

# Построение словаря и кодирование текста
class Vocabulary:
    def __init__(self, pad_token="<pad>", unk_token="<unk>", min_freq=1):
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.min_freq = min_freq
        
        self.w2i = {self.pad_token: 0, self.unk_token: 1}
        self.i2w = {0: self.pad_token, 1: self.unk_token}
        
    def build_vocab(self, texts):
        counter = collections.Counter()
        for text in texts:
            counter.update(text.split())
            
        for word, freq in counter.items():
            if freq >= self.min_freq and word not in self.w2i:
                idx = len(self.w2i)
                self.w2i[word] = idx
                self.i2w[idx] = word
                
    def encode(self, text):
        tokens = text.split()
        return [self.w2i.get(token, self.w2i[self.unk_token]) for token in tokens]

    def __len__(self):
        return len(self.w2i)

# Инициализация и сборка словаря по обучающей выборке
vocab = Vocabulary(min_freq=1)
vocab.build_vocab(X_train)
print(f"Размер словаря: {len(vocab)} токенов")

Размер словаря: 141 токенов


In [236]:
class BankingDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.labels = list(labels)
        self.encoded_texts = [vocab.encode(t) for t in texts]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded_texts[idx], self.labels[idx]

def collate_fn(batch):
    text_list, label_list, offsets= [], [], [0]
    for _text, _label in batch:
        label_list.append(_label)
        processed_text =torch.tensor(_text, dtype=torch.long)
        text_list.append(processed_text)
        offsets.append(processed_text.size(0))
        
    label_list = torch.tensor(label_list, dtype=torch.long)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    text_list = torch.cat(text_list)
    
    return text_list, offsets, label_list

# Создание DataLoader
train_dataset = BankingDataset(X_train, y_train, vocab)
test_dataset = BankingDataset(X_test, y_test, vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

In [237]:
class BankingTextMLP(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode='mean')
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.relu =nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        x = self.fc1(embedded)
        x = self.relu(x)
        x = self.dropout(x)
        out =self.fc2(x)
        return out

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = BankingTextMLP(
    vocab_size=len(vocab),
    embed_dim=128,
    hidden_dim =64,
    num_classes=len(label_mapping),
    dropout=0.3
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

In [238]:
epochs = 100

for epoch in range(1, epochs + 1):
    # Обучение
    model.train()
    total_loss = 0.0
    for text, offsets, labels in train_loader:
        text, offsets, labels = text.to(device), offsets.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(text, offsets)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    # Валидация
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for text, offsets, labels in test_loader:
            text, offsets, labels= text.to(device), offsets.to(device), labels.to(device)
            outputs = model(text, offsets)
            preds = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

print("\nПодробный отчет для PyTorch BankingTextMLP:")
print(classification_report(all_targets, all_preds, target_names=[inv_label_mapping[i] for i in range(len(label_mapping))]))


Подробный отчет для PyTorch BankingTextMLP:
               precision    recall  f1-score   support

        Карты       0.96      0.99      0.98       100
       Сервис       1.00      1.00      1.00        95
Мошенничество       0.99      0.98      0.98       100
      Кредиты       0.94      0.93      0.94       105
     Переводы       0.90      0.89      0.89       100

     accuracy                           0.96       500
    macro avg       0.96      0.96      0.96       500
 weighted avg       0.96      0.96      0.96       500

